In [0]:
import pandas as pd
from typing import Iterator
from pyspark.sql.functions import pandas_udf, col, spark_partition_id, asc, create_map, array, lit
from pyspark.sql.types import *
import time
from datetime import date
import random
from faker import Faker
from mimesis import Generic
from mimesis.locales import Locale

schema = StructType([
  StructField("customer_id", LongType(), False),
  StructField("name", StringType(), False),
  StructField("email", StringType(), False),
  StructField("date_of_birth", DateType(), False),
  StructField("age", LongType(), False),
  StructField("address", StringType(), False),
  StructField("postcode", StringType(), False),
  StructField("ipv4", StringType(), False),
  StructField("ipv4_with_port", StringType(), False),
  StructField("ipv6", StringType(), False),
  StructField("mac_address", StringType(), False),
  StructField("phone_number", StringType(), False),
  StructField("ssn", StringType(), False),
  StructField("itin", StringType(), False),
  StructField("iban", StringType(), False),
  StructField("credit_card", LongType(), False),
  StructField("credit_card_with_spaces", StringType(), False),
  StructField("credit_card_full", StringType(), False),
  StructField("expiry_date", StringType(), False),
  StructField("security_code", StringType(), False),
  StructField("freetext", StringType(), False),
  StructField("passport", StringType(), False),
  StructField("aba", StringType(), False),
  StructField("bban", StringType(), False),
  StructField("uri", StringType(), False),
  StructField("url", StringType(), False),
  StructField("language", StringType(), False),
  StructField("nationality", StringType(), False),
  StructField("country", StringType(), False),
  StructField("date_time", StringType(), False),
  ])

fake = Faker("en_US")
generic = Generic(locale=Locale.EN, seed=1)

def get_random_pii():
  return random.choice([fake.ascii_free_email(), fake.ipv4(), fake.ipv6()])

@pandas_udf("long")
def get_customer_id(batch_iter: Iterator[pd.Series]) -> Iterator[pd.Series]:
  for id in batch_iter:
      yield int(time.time()) + id

pii_struct_schema = StructType([
    StructField("email_address", StringType(), False),
    StructField("ipv4_private", StringType(), False),
    StructField("ip_address_v6", StringType(), False),
    StructField("ipv4_with_port", StringType(), False),
    StructField("mac", StringType(), False),
    StructField("imei", StringType(), False),
    StructField("credit_card_number", StringType(), False), 
    StructField("credit_card_expiration_date", StringType(), False), 
    StructField("cvv", StringType(), False), 
    StructField("paypal", StringType(), False), 
    StructField("random_text_with_email", StringType(), False),
    StructField("random_text_with_ipv4", StringType(), False)
])

def pii_struct():
  return (generic.person.email(), fake.ipv4_private(), fake.ipv6(), generic.internet.ip_v4_with_port(), generic.internet.mac_address(), generic.code.imei(), generic.payment.credit_card_number(), generic.payment.credit_card_expiration_date(), generic.payment.cvv(), generic.payment.paypal(), f"{fake.catch_phrase()} {generic.person.email()}", f"{fake.catch_phrase()} {fake.ipv4_public()}")

pii_struct_udf = udf(pii_struct, pii_struct_schema)

def generate_fake_data(pdf: pd.DataFrame) -> pd.DataFrame:
    
  def generate_data(y):
    
    dob = fake.date_between(start_date='-99y', end_date='-18y')

    y["name"] = fake.name()
    y["email"] = fake.ascii_free_email()
    y["date_of_birth"] = dob #.strftime("%Y-%m-%d")
    y["age"] = date.today().year - dob.year
    y["address"] = fake.address()
    y["ipv4"] = fake.ipv4()
    y["ipv4_with_port"] = generic.internet.ip_v4_with_port()
    y["ipv6"] = fake.ipv6()
    y["mac_address"] = fake.mac_address()
    y["postcode"] = fake.postcode()
    y["phone_number"] = fake.phone_number()
    y["ssn"] = fake.ssn()
    y["itin"] = fake.itin()
    y["iban"] = fake.iban()
    y["credit_card"] = int(fake.credit_card_number())
    y["credit_card_with_spaces"] = generic.payment.credit_card_number()
    y["credit_card_full"] = fake.credit_card_full()
    y["expiry_date"] = fake.credit_card_expire()
    y["security_code"] = fake.credit_card_security_code()
    y["freetext"] = f"{fake.sentence()} {get_random_pii()} {fake.sentence()} {get_random_pii()} {fake.sentence()}"
    y["passport"] = fake.passport_number()
    y["aba"] = fake.aba()
    y["bban"] = fake.bban()
    y["uri"] = fake.uri()
    y["url"] = fake.url()
    y["language"] = generic.person.language()
    y["nationality"] = generic.person.nationality()
    y["country"] = fake.country()
    y["date_time"] = fake.date_time().strftime("%c")

    return y
    
  return pdf.apply(generate_data, axis=1).drop(["partition_id", "id"], axis=1)

In [0]:
def generate_fake_pii_data(num_rows=1000):
  initial_data = spark.range(1, num_rows+1).withColumn("customer_id", get_customer_id(col("id"))) 
  return (
  initial_data
  .withColumn("partition_id", spark_partition_id())
  .groupBy("partition_id")
  .applyInPandas(generate_fake_data, schema)
  .withColumn("pii_struct", pii_struct_udf())
  .withColumn("pii_map", create_map(lit("email_address"), col("email"), lit("ip_address"), col("ipv4"), lit("home_address"), col("address"))) 
  .withColumn("pii_array", array("email", "ipv4", "ipv6"))
  .orderBy(asc("customer_id")))

In [0]:
def get_selection(selection: list, all_options: list) -> list:

  if "ALL" in selection:
    return all_options
  else:
    return selection

In [0]:
# See https://microsoft.github.io/presidio/supported_entities/ 

all_supported_entities = ["CREDIT_CARD", "CRYPTO", "DATE_TIME", "EMAIL_ADDRESS", "IBAN_CODE", "IP_ADDRESS", "NRP", "LOCATION", "PERSON", "PHONE_NUMBER", "MEDICAL_LICENSE", "URL", "US_BANK_NUMBER", "US_DRIVER_LICENSE", "US_ITIN", "US_PASSPORT", "US_SSN", "UK_NHS", "ES_NIF", "IT_FISCAL_CODE", "IT_DRIVER_LICENSE", "IT_VAT_CODE", "IT_PASSPORT", "IT_IDENTITY_CARD", "SG_NRIC_FIN", "AU_ABN", "AU_ACN", "AU_TFN", "AU_MEDICARE"]

In [0]:
from pyspark.sql import SparkSession, DataFrame
from pyspark.broadcast import Broadcast
from pyspark.sql.functions import asc, col, when, lit, from_json, explode, mean, count, pandas_udf
from pyspark.sql.types import StringType, ArrayType, StructType, StructField, IntegerType, DoubleType
import pandas as pd
import json
from datetime import date

class PIIScanner:

  def __init__(
    self, 
    spark: SparkSession, broadcasted_analyzer: Broadcast, entities: list, 
    language: str = "en",  sample_size: int = 1000, average_score: float = 0.5, hit_rate: int = 60):
      self.spark = spark 
      self.broadcasted_analyzer = broadcasted_analyzer
      self.entities = entities
      self.language = language
      self.sample_size = sample_size
      self.average_score = average_score
      self.hit_rate = hit_rate
      self.scan_schema = ArrayType(StructType([
        StructField("entity_type", StringType()),
        StructField("start", IntegerType()),
        StructField("end", IntegerType()),
        StructField("score", DoubleType())
      ]))
      self.results_schema = StructType([
        StructField("column", StringType()),
        StructField("entity_type", StringType()),
        StructField("num_entities", DoubleType()),
        StructField("avg_score", DoubleType()),
        StructField("sample_size", IntegerType()),
        StructField("hit_rate", DoubleType()),
    ])
      print(f"PII Scanner initialized using language {self.language.upper()}. Looking for entities {self.entities} with a sample size of {self.sample_size}, an average score of {self.average_score} and a hit rate of {self.hit_rate}.")

  @staticmethod
  def get_all_uc_tables(spark: SparkSession, catalogs: tuple) -> DataFrame:

    sql_clause = None
    print(f"Getting all uc tables in catalogs {', '.join(catalogs)}")
    if len(catalogs) == 1:
      sql_clause = f"table_catalog IN ('{catalogs[0]}')"
    else:
      sql_clause = f"table_catalog IN {catalogs}"
    return (
      spark.sql(f"SELECT * FROM system.information_schema.tables WHERE {sql_clause}")
      .select("table_catalog", 
              "table_schema", 
              "table_name",
              when(col("table_type") == "VIEW", "VIEW").otherwise(lit("TABLE")).alias("table_type"),
              "created", 
              "last_altered").
      orderBy(
        col("table_catalog").asc(), 
        col("table_schema").asc(), 
        col("table_name").asc()))

  def _get_aggregated_results(self, df: DataFrame) -> DataFrame:

    df_size = df.count()

    if df_size != self.sample_size:
      sample_size = df_size
    else:
      sample_size = self.sample_size
    results_df = spark.createDataFrame([], self.results_schema)

    for c in df.columns:
      
      exploded = df.select(lit(c).alias("column"), explode(col(c)).alias(c))
      new_df = (exploded.select(
        col("column"),
        col(f"{c}.entity_type"), 
        col(f"{c}.score")).groupBy("column", "entity_type")
                .agg(
                  count("entity_type").alias("num_entities"),
                  mean("score").alias("avg_score"))
                .withColumn("sample_size", lit(sample_size))
                .withColumn("hit_rate", col("num_entities") / col("sample_size") * 100)
                .where(f"hit_rate >= {self.hit_rate} AND avg_score >= {self.average_score}"))
      results_df = results_df.union(new_df)

    return results_df

  def _scan_dataframe(self, df: DataFrame) -> DataFrame:
    return (df.select([from_json(analyze_udf(col(c).cast("string")), self.scan_schema).alias(c) for c in df.columns]))

  def scan_dataframe(self, df: DataFrame) -> DataFrame:

    try:
      scanned = self._scan_dataframe(df).limit(self.sample_size)
      results = self._get_aggregated_results(scanned)

    except Exception as e:
      print(f"Failed to scan {securable_type} {securable_namespace} because of exception {e}")

    return results
  
  def add_tag(self, securable_namespace: str, securable_type: str, tag: str, column: str=None) -> None:

    sql_query, log_message = None, None
    if column:
      sql_query = f"ALTER {securable_type} {securable_namespace} ALTER COLUMN {column} SET TAGS ('{tag}')"
      log_message = f"Adding tag '{tag}' to column {column} on {securable_type} {securable_namespace}"
    else:
      sql_query = f"ALTER {securable_type} {securable_namespace} SET TAGS ('{tag}')"
      log_message = f"Adding tag '{tag}' to {securable_type} {securable_namespace}"
    
    try:
      print(log_message)
      self.spark.sql(sql_query)
    except Exception as e:
      print(f"Failed to add tag '{tag}' to {securable_type} {securable_namespace} because of exception {e}")

  def add_comment(self, securable_namespace: str, securable_type: str, comment: str, column: str=None) -> None:

    sql_query, log_message = None, None
    if column:
      sql_query = f"ALTER {securable_type} {securable_namespace} ALTER COLUMN {column} COMMENT '{comment}'"
      log_message = f"Adding comment to column {column} on {securable_type} {securable_namespace}"
    else:
      sql_query = f"COMMENT ON {securable_type} {securable_namespace} IS '{comment}'"
      log_message = f"Adding comment to {securable_type} {securable_namespace}"

    try:
      print(log_message)
      self.spark.sql(sql_query)
    
    except Exception as e:
      print(f"Failed to add comment to {securable_type} {securable_namespace} because of exception {e}")
  
  def _get_table_comment(self, date: date) -> str:

    return f"""> # `WARNING! This table contains PII`
# Table Scanned on `{date}`"""

  def _get_column_comment(self, column_json) -> str:

    return f"""> # WARNING! This column contains PII:
```
{json.dumps(column_json)}
```
"""

  def scan_and_tag_securable(self, securable_namespace: str, securable_type: str) -> DataFrame:

    today = date.today()
    print(f"Scanning {securable_type} {securable_namespace} for PII.")

    df = self.spark.table(securable_namespace).limit(self.sample_size)
    scanned = self._scan_dataframe(df)
    results = self._get_aggregated_results(scanned).toPandas()

    if len(results) > 0:

      self.add_tag(securable_namespace=securable_namespace, securable_type=securable_type, tag='PII')
      if securable_type == "TABLE":
        self.add_comment(securable_namespace=securable_namespace, securable_type=securable_type, comment=self._get_table_comment(today))

      for index, value in results["column"].drop_duplicates().items():

        result = results[results["column"] == value] 
        column_json = []

        for index, row in result.iterrows():
          self.add_tag(securable_namespace, securable_type, row.entity_type)
          column_json.append(row.to_json(indent=2))
          if securable_type == "TABLE":
            self.add_tag(securable_namespace=securable_namespace, securable_type=securable_type, tag=row.entity_type, column=row.column)
        if securable_type == "TABLE":
          self.add_comment(securable_namespace=securable_namespace, securable_type=securable_type, comment=self._get_column_comment(column_json), column=row.column)
    results.insert(0, "scan_date", today)
    results.insert(1, "securable", securable_namespace)
    return results
  
  def get_pii_tagged_columns(self, catalog: str, schema: str, table: str) -> list[tuple[str, str|None]]:
      """
      Return [(column_name, entity_or_None)] for columns tagged as PII or with one
      of the Presidio entity tags present in self.entities.
      """
      entity_set = set(self.entities)  # e.g., EMAIL_ADDRESS, PERSON, URL, DATE_TIME, ...
      sql_text = f"""
        SELECT column_name, tag_name
        FROM system.information_schema.column_tags
        WHERE catalog_name = '{catalog}'
          AND schema_name  = '{schema}'
          AND table_name   = '{table}'
          AND tag_name IS NOT NULL
      """
      rows = self.spark.sql(sql_text).collect()

      # DEBUG: see what tags we actually found
      if not rows:
          print(f"[DEBUG] No column_tags rows for {catalog}.{schema}.{table}")
      else:
          print(f"[DEBUG] column_tags for {catalog}.{schema}.{table}: " +
                ", ".join([f"{r['column_name']}={r['tag_name']}" for r in rows]))

      pii_cols = []
      for r in rows:
          tag = r["tag_name"]
          col = r["column_name"]
          if tag == "PII":
              pii_cols.append((col, None))             # generic → use all entities on this col
          elif tag in entity_set:
              pii_cols.append((col, tag))              # specific entity → narrow to that entity

      # Prefer specific entity over generic when duplicates exist
      merged = {}
      for c, ent in pii_cols:
          merged[c] = merged.get(c, ent) or ent

      result = [(c, e) for c, e in merged.items()]

      print(f"[DEBUG] get_pii_tagged_columns -> {len(result)} column(s): {result}")
      return result


  def _anonymize_series(self, series, active_entities, language):
      from presidio_anonymizer import AnonymizerEngine
      analyzer = self.broadcasted_analyzer.value
      anonymizer = AnonymizerEngine()
      from presidio_anonymizer.operators import OperatorConfig

      DEFAULT_OPERATOR_CONFIG = {
          "EMAIL_ADDRESS": OperatorConfig("replace", {"new_value": "<EMAIL>"}),
          "PHONE_NUMBER": OperatorConfig("replace", {"new_value": "<PHONE>"}),
          "PERSON": OperatorConfig("replace", {"new_value": "<PERSON>"}),
          "CREDIT_CARD": OperatorConfig("mask", {"new_value": "*", "from_end": 4}),
      }

      def _anon(x):
          if not x:
              return x
          results = analyzer.analyze(text=str(x), entities=active_entities, language=language)
          if not results:
              return x
          return anonymizer.anonymize(
              text=str(x),
              analyzer_results=results,
              operators={e: DEFAULT_OPERATOR_CONFIG.get(e, OperatorConfig("replace", {"new_value": "<PII>"})) for e in active_entities}
          ).text
      return series.astype(str).apply(_anon)

  def _anonymize_table(self, catalog: str, schema: str, table: str, anon_suffix="_anonymized"):
    df = self.spark.table(f"{catalog}.{schema}.{table}")

    # Use the fixed column-tag fetcher you just added
    pii_cols = self.get_pii_tagged_columns(catalog, schema, table)
    if not pii_cols:
        print(f"[SKIP] No PII columns in {catalog}.{schema}.{table}")
        return None

    per_col_entities = {c: ([e] if e else list(self.entities)) for c, e in pii_cols}

    out_cols = []
    from pyspark.sql.functions import col

    for c in df.columns:
        if c in per_col_entities:
            ents = per_col_entities[c]
            op_cfg = _build_ops(ents)  # <-- now returns OperatorConfig objects
            col_udf = make_anonymize_pandas_udf(ents, self.language, op_cfg)
            out_cols.append(col_udf(col(c).cast("string")).alias(c))
        else:
            out_cols.append(col(c))


    anon_df = df.select(*out_cols)
    out_table = f"{catalog}.{schema}.{table}{anon_suffix}"
    anon_df.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable(out_table)
    print(f"[CREATED] {out_table}")
    return out_table


  def anonymize_all_tagged_tables_in_catalog(self, catalog: str, anon_suffix: str = "_anonymized", overwrite: bool = True, dry_run: bool = False):
      # Build entity list from the class field (NOT a free variable)
      entity_list_sql = ", ".join([f"'{e}'" for e in self.entities]) or "''"

      candidate_sql = f"""
        WITH tagged_cols AS (
          SELECT DISTINCT
            schema_name AS table_schema,
            table_name
          FROM system.information_schema.column_tags
          WHERE catalog_name = '{catalog}'
            AND (tag_name = 'PII' OR tag_name IN ({entity_list_sql}))
        ),
        tagged_tables AS (
          SELECT DISTINCT
            schema_name AS table_schema,
            table_name
          FROM system.information_schema.table_tags
          WHERE catalog_name = '{catalog}'
            AND tag_name = 'PII'
        ),
        candidates AS (
          SELECT * FROM tagged_cols
          UNION
          SELECT * FROM tagged_tables
        )
        SELECT b.table_schema, b.table_name, b.table_type
        FROM candidates t
        INNER JOIN system.information_schema.tables b
          ON b.table_catalog = '{catalog}'
        AND b.table_schema  = t.table_schema
        AND b.table_name    = t.table_name
        WHERE b.table_schema <> 'information_schema'   -- avoid IS objects
        -- NOTE: no table_type filter; 'MANAGED' is valid here
      """


      print(f"[INFO] Finding PII-tagged tables in catalog: {catalog}")
      print(f"[DEBUG] Entities used for selection: {self.entities}")
      candidates = self.spark.sql(candidate_sql).collect()
      print(f"[INFO] Found {len(candidates)} candidate table(s)")

      created, skipped = [], []

      for row in candidates:
          schema = row["table_schema"]
          table = row["table_name"]
          fq = f"{catalog}.{schema}.{table}"
          try:
              if dry_run:
                  print(f"[DRY-RUN] Would anonymize: {fq}")
                  continue

              # Reuse your single-table anonymizer
              out = self._anonymize_table(catalog, schema, table, anon_suffix=anon_suffix)
              if out:
                  created.append(out)
              else:
                  skipped.append((schema, table, "no_pii_columns"))
          except Exception as e:
              skipped.append((schema, table, f"error: {e}"))

      print(f"\n=== Anonymization summary for catalog '{catalog}' ===")
      print(f"Created: {len(created)}")
      for t in created:
          print(f"  - {t}")
      print(f"Skipped: {len(skipped)}")
      for s in skipped:
          print(f"  - {s[0]}.{s[1]} ({s[2]})")

      return created, skipped


In [0]:
from presidio_anonymizer.entities import OperatorConfig  # <-- important
from presidio_anonymizer import AnonymizerEngine

FALLBACK_OP = OperatorConfig("replace", {"new_value": "<PII>"})
broadcasted_anonymizer = sc.broadcast(AnonymizerEngine())

def _build_ops(active_entities, overrides=None):
    """
    Returns dict[str, OperatorConfig] for AnonymizerEngine.
    Accepts optional 'overrides' in a few forms and normalizes them to OperatorConfig.
    """
    defaults = {
        "EMAIL_ADDRESS": OperatorConfig("replace", {"new_value": "<EMAIL>"}),
        "IP_ADDRESS":    OperatorConfig("replace", {"new_value": "<IP>"}),
        "URL":           OperatorConfig("replace", {"new_value": "<URL>"}),
        "DATE_TIME":     OperatorConfig("replace", {"new_value": "<DATE>"}),
        "PERSON":        OperatorConfig("replace", {"new_value": "<PERSON>"}),
        "LOCATION":      OperatorConfig("replace", {"new_value": "<LOCATION>"}),
        # add more if you like; fallback handles the rest
    }

    def _to_operator_config(val):
        # Already an OperatorConfig
        if isinstance(val, OperatorConfig):
            return val
        # Allow {"operator_name": "...", "params": {...}}
        if isinstance(val, dict) and "operator_name" in val:
            return OperatorConfig(val["operator_name"], val.get("params") or {})
        # Allow {"type": "replace"|"mask"|..., <params...>}
        if isinstance(val, dict) and "type" in val:
            op = val["type"]
            params = {k: v for k, v in val.items() if k != "type"}
            # if a mask override came in using "new_value", map to "masking_char"
            if op == "mask" and "new_value" in params and "masking_char" not in params:
                params["masking_char"] = params.pop("new_value")
            return OperatorConfig(op, params)
        # Fallback
        return FALLBACK_OP

    ops = {}
    for e in active_entities:
        if overrides and e in overrides:
            ops[e] = _to_operator_config(overrides[e])
        else:
            ops[e] = defaults.get(e, FALLBACK_OP)
    return ops
def make_anonymize_pandas_udf(active_entities, language, operator_cfg):
    """Factory that returns a pandas_udf WITHOUT capturing `self` or `spark`."""
    analyzer_bc = broadcasted_analyzer     # module-level var from your notebook
    anonymizer_bc = broadcasted_anonymizer # module-level var defined above

    def _series(s: pd.Series) -> pd.Series:
        analyzer = analyzer_bc.value
        anonymizer = anonymizer_bc.value
        ents = list(active_entities)  # freeze for the closure
        ops = dict(operator_cfg)

        def _anon_one(x):
            if x is None:
                return None
            text = str(x)
            res = analyzer.analyze(text=text, entities=ents, language=language)
            if not res:
                return text
            return anonymizer.anonymize(text=text, analyzer_results=res, operators=ops).text

        return s.astype(object).apply(_anon_one)

    return pandas_udf(_series, returnType=StringType())
